In [1]:
import pickle
import time

from omnes_pro_uno.fphse import Fphse

OP_ADD = b"\x01"

with open("../lib/enron_kw100_00-01_timeline.pkl", "rb") as f:
    enron = pickle.load(f)
print("enron loaded")

data_owner_num = len(enron.keys())
writers = list(enron.keys())

# Setup
setup_t = time.time()

fphse = Fphse(data_owner_num)
msk = fphse.rsetup()

wk_vec = []
e_tkn_vec = []
st_vec = []
enb_vec = []
b_vec = []


def setup_data_owner(i):
    b_vec.append({})
    wk, st, enb = fphse.wsetup()
    e_tkn, st = fphse.rebuild(i, wk, b_vec[i], st)
    wk_vec.append(wk)
    e_tkn_vec.append(e_tkn)
    st_vec.append(st)
    enb_vec.append(enb)


# Data owner setup can be parallelized
setup_data_owner(0)

# Setup done
print(f"setup (s): {time.time() - setup_t}")

for i in range(1, data_owner_num):
    setup_data_owner(i)

# Update
update_t = time.time()
update_epoch_t = time.time()


def epoch_rotation():
    fphse.edsse.set_epoch(fphse.edsse.get_epoch() + 1)
    for i in range(data_owner_num):
        e_tkn_vec[i], st_vec[i] = fphse.rebuild(i, wk_vec[i], b_vec[i], st_vec[i])


epoch = 0
total_update_d = 0.0

for year in ["2000", "2001"]:
    for month in range(1, 13):
        if month < 10:
            month_s = "0" + str(month)
        else:
            month_s = str(month)
        date = year + "-" + month_s

        for i, writer in enumerate(writers):
            map = enron[writer]
            posts = map[date]
            for post in posts:
                pid = post["id"].encode()
                if len(pid) > 47:
                    pid = pid[:47]
                else:
                    pid = pid + b"\0" * (47 - len(pid))

                for kw in post["keywords"]:
                    u_no_sse, st_vec[i] = fphse.update_token(
                        i, b_vec[i], wk_vec[i], st_vec[i], OP_ADD, kw.encode(), pid
                    )
                    enb_vec[i], e_tkn_vec[i] = fphse.update(u_no_sse, enb_vec[i], e_tkn_vec[i])
        epoch_rotation()

        print(f"epoch {epoch} done (s): {time.time() - update_epoch_t}")
        update_epoch_t = time.time()
        epoch += 1

# Update done
print(f"all epoch done (s): {time.time() - update_t}")

enron loaded
setup (s): 0.16651344299316406
epoch 0 done (s): 61.12783432006836
epoch 1 done (s): 42.39343976974487
epoch 2 done (s): 46.10644316673279
epoch 3 done (s): 51.64558005332947
epoch 4 done (s): 55.325745582580566
epoch 5 done (s): 79.56123423576355
epoch 6 done (s): 73.1297538280487
epoch 7 done (s): 78.56159448623657
epoch 8 done (s): 83.31964063644409
epoch 9 done (s): 95.71567487716675
epoch 10 done (s): 115.48981952667236
epoch 11 done (s): 123.47824168205261
epoch 12 done (s): 127.10531663894653
epoch 13 done (s): 130.09300541877747
epoch 14 done (s): 135.26743721961975
epoch 15 done (s): 155.30824065208435
epoch 16 done (s): 173.81461834907532
epoch 17 done (s): 169.98690056800842
epoch 18 done (s): 173.38877177238464
epoch 19 done (s): 174.8561520576477
epoch 20 done (s): 188.73285245895386
epoch 21 done (s): 200.17527222633362
epoch 22 done (s): 192.4183189868927
epoch 23 done (s): 192.80046558380127
all epoch done (s): 2919.820771217346


In [8]:
s_i = writers.index("davis-d")
w = b'dan'

In [10]:
s_num_vec = [5, 10, 15, 20, 25]
s_vec = [list(range(i)) for i in s_num_vec]
for idx, s in enumerate(s_vec):
    if s_i not in s:
        s.pop()
        s.append(s_i)
    assert len(s) == s_num_vec[idx]

    search_token_t = time.time()
    s_no_sse = fphse.search_token(msk, s, w)
    print(f"search token (s): {time.time() - search_token_t}")

    search_t = time.time()
    r, _, _ = fphse.search(s_no_sse, s, enb_vec, e_tkn_vec)
    print(f"search (s): {time.time() - search_t}")
    print(len(r))

search token (s): 0.0011343955993652344
Writer 0 time (s): 0.5488376617431641
Writer 1 time (s): 0.5450761318206787
Writer 2 time (s): 0.5438389778137207
Writer 3 time (s): 0.5555324554443359
Writer 11 time (s): 0.5531620979309082
search (s): 2.7469491958618164
824
search token (s): 0.001209259033203125
Writer 0 time (s): 0.5496740341186523
Writer 1 time (s): 0.5541174411773682
Writer 2 time (s): 0.5590143203735352
Writer 3 time (s): 0.5540611743927002


KeyboardInterrupt: 

In [6]:
epoch_rotation_t = time.time()
epoch_rotation()
print(f"epoch rotation (s): {time.time() - epoch_rotation_t}")

epoch rotation (s): 194.08063864707947
